# A/B Testing: Publicidad estática (A) vs. Publicidad variante (B)

**Caso de estudio**

Una tienda online quiere decidir qué anuncio estático (banner) usar en su sitio web:

- **Anuncio A (base / control):** el anuncio actual, con el que ya se tiene histórico.
- **Anuncio B (variante / tratamiento):** un nuevo diseño propuesto.

Cada visitante ve aleatoriamente uno de los dos anuncios. Medimos si el visitante hace **clic** (y opcionalmente si **convierte**, es decir compra). El objetivo es responder: **¿el Anuncio B tiene una tasa de clics/conversión significativamente distinta a la del Anuncio A?**

Este notebook:
1. Simula datos realistas de un experimento A/B (impresiones, clics, conversiones).
2. Calcula métricas descriptivas (CTR, tasa de conversión) por grupo.
3. Visualiza los resultados con intervalos de confianza.
4. Aplica pruebas de hipótesis (z-test de proporciones y chi-cuadrado) para determinar significancia estadística.
5. Da una conclusión y recomendación de negocio.

> Nota: los datos son **simulados**. Para usar datos reales, sustituye la celda de simulación por la carga de tu propio CSV (impresiones, clics, conversiones por visitante o agregados por grupo).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Simulación del experimento

Simulamos visitantes que ven el Anuncio A (base) o el Anuncio B (variante). Cada visitante:
- Hace **clic** en el anuncio o no (evento binario, modelado con una tasa de clics real "desconocida" `p_click`).
- Si hace clic, puede **convertir** (comprar) o no (tasa de conversión condicional al clic, `p_convert_given_click`).

Supondremos que el Anuncio B tiene una mejora real pequeña en CTR (esto es lo que la prueba estadística debe detectar o no, según el tamaño de muestra).

In [ ]:
# Parámetros "reales" de la simulación (desconocidos en un experimento real)
n_a = 5000          # impresiones del Anuncio A (base)
n_b = 5000          # impresiones del Anuncio B (variante)

p_click_a = 0.08            # CTR real del Anuncio A
p_click_b = 0.095           # CTR real del Anuncio B (ligera mejora)

p_convert_given_click_a = 0.10   # tasa de conversión condicional al clic, Anuncio A
p_convert_given_click_b = 0.11   # tasa de conversión condicional al clic, Anuncio B

# Simular clics (1 = clic, 0 = no clic) por visitante
clicks_a = np.random.binomial(1, p_click_a, n_a)
clicks_b = np.random.binomial(1, p_click_b, n_b)

# Simular conversiones solo entre quienes hicieron clic
conversions_a = np.where(clicks_a == 1, np.random.binomial(1, p_convert_given_click_a, n_a), 0)
conversions_b = np.where(clicks_b == 1, np.random.binomial(1, p_convert_given_click_b, n_b), 0)

df = pd.DataFrame({
    'grupo': ['A'] * n_a + ['B'] * n_b,
    'anuncio': ['Anuncio A (base)'] * n_a + ['Anuncio B (variante)'] * n_b,
    'click': np.concatenate([clicks_a, clicks_b]),
    'conversion': np.concatenate([conversions_a, conversions_b]),
})

df.head()

## 2. Métricas descriptivas por grupo

In [ ]:
resumen = df.groupby('anuncio').agg(
    impresiones=('click', 'size'),
    clics=('click', 'sum'),
    conversiones=('conversion', 'sum'),
)
resumen['ctr'] = resumen['clics'] / resumen['impresiones']
resumen['tasa_conversion'] = resumen['conversiones'] / resumen['impresiones']
resumen['conversion_por_clic'] = resumen['conversiones'] / resumen['clics']

resumen

## 3. Visualización con intervalos de confianza (95%)

Comparamos el CTR y la tasa de conversión de cada anuncio, con su intervalo de confianza del 95% (método de Wilson), para ver qué tan solapados están los resultados.

In [ ]:
def plot_metric_with_ci(df, metric_col, title, ylabel):
    grupos = ['Anuncio A (base)', 'Anuncio B (variante)']
    tasas = []
    errores_inf = []
    errores_sup = []

    for g in grupos:
        sub = df[df['anuncio'] == g]
        exitos = sub[metric_col].sum()
        n = len(sub)
        tasa = exitos / n
        ci_low, ci_high = proportion_confint(exitos, n, alpha=0.05, method='wilson')
        tasas.append(tasa)
        errores_inf.append(tasa - ci_low)
        errores_sup.append(ci_high - tasa)

    fig, ax = plt.subplots()
    colores = ['#6366f1', '#22c55e']
    ax.bar(grupos, tasas, yerr=[errores_inf, errores_sup], capsize=8, color=colores)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    for i, t in enumerate(tasas):
        ax.text(i, t + errores_sup[i] + 0.002, f"{t:.2%}", ha='center', fontweight='bold')
    plt.show()

plot_metric_with_ci(df, 'click', 'CTR por anuncio (IC 95%)', 'CTR (tasa de clics)')
plot_metric_with_ci(df, 'conversion', 'Tasa de conversión por anuncio (IC 95%)', 'Tasa de conversión')

## 4. Prueba de hipótesis

**Hipótesis nula (H0):** no hay diferencia real en la tasa de clics (o conversión) entre el Anuncio A y el Anuncio B.

**Hipótesis alternativa (H1):** las tasas son diferentes.

Usamos:
- **Z-test de proporciones** (`proportions_ztest`) para comparar directamente las dos proporciones.
- **Chi-cuadrado de independencia** como verificación alternativa sobre la tabla de contingencia.

Nivel de significancia: `alpha = 0.05`.

In [ ]:
alpha = 0.05

def evaluar_metrica(df, metric_col, nombre_metrica):
    exitos = df.groupby('anuncio')[metric_col].sum()
    n = df.groupby('anuncio')[metric_col].count()

    exitos_vec = np.array([exitos['Anuncio A (base)'], exitos['Anuncio B (variante)']])
    n_vec = np.array([n['Anuncio A (base)'], n['Anuncio B (variante)']])

    z_stat, p_value = proportions_ztest(exitos_vec, n_vec, alternative='two-sided')

    tabla = pd.crosstab(df['anuncio'], df[metric_col])
    chi2, p_chi2, dof, expected = stats.chi2_contingency(tabla)

    print(f"--- {nombre_metrica} ---")
    print(f"Tasa A: {exitos_vec[0] / n_vec[0]:.4f}  |  Tasa B: {exitos_vec[1] / n_vec[1]:.4f}")
    print(f"Z-test de proporciones -> z = {z_stat:.3f}, p-valor = {p_value:.4f}")
    print(f"Chi-cuadrado           -> chi2 = {chi2:.3f}, p-valor = {p_chi2:.4f}")

    if p_value < alpha:
        print(f"Resultado: diferencia ESTADÍSTICAMENTE SIGNIFICATIVA (p < {alpha}).")
    else:
        print(f"Resultado: NO hay evidencia suficiente de diferencia (p >= {alpha}).")
    print()

    return p_value

p_click = evaluar_metrica(df, 'click', 'CTR (clics / impresiones)')
p_conv = evaluar_metrica(df, 'conversion', 'Tasa de conversión (conversiones / impresiones)')

## 5. Conclusión y recomendación

In [ ]:
ctr_a = resumen.loc['Anuncio A (base)', 'ctr']
ctr_b = resumen.loc['Anuncio B (variante)', 'ctr']
mejora_relativa = (ctr_b - ctr_a) / ctr_a * 100

print(f"CTR Anuncio A (base):     {ctr_a:.2%}")
print(f"CTR Anuncio B (variante): {ctr_b:.2%}")
print(f"Mejora relativa de B sobre A: {mejora_relativa:+.1f}%")
print()

if p_click < alpha and ctr_b > ctr_a:
    print("Recomendación: Adoptar el Anuncio B. La mejora en CTR es estadísticamente "
          "significativa y positiva frente al Anuncio A (base).")
elif p_click < alpha and ctr_b < ctr_a:
    print("Recomendación: Mantener el Anuncio A (base). El Anuncio B tiene un CTR "
          "significativamente más bajo.")
else:
    print("Recomendación: No hay evidencia suficiente para afirmar que el Anuncio B "
          "sea mejor o peor que el Anuncio A. Se sugiere aumentar el tamaño de muestra "
          "o la duración del experimento antes de decidir.")

## 6. Usar tus propios datos

Para reemplazar la simulación por datos reales, carga un CSV con una fila por visitante (o por grupo agregado) con columnas `grupo` (`A`/`B`), `click` (0/1) y `conversion` (0/1), por ejemplo:

```python
df = pd.read_csv('mi_experimento.csv')
```

y vuelve a ejecutar las celdas desde la sección 2 en adelante.